# Extragere date StatsBomb Open Data

Notebook-ul descarcă datele publicate în depozitul GitHub `hudl/open-data`: competiții, meciuri, evenimente și line-up-uri. Selectează o competiție și un sezon disponibile, apoi salvează răspunsurile brute JSON și tabele CSV. Acoperirea este limitată la competițiile publicate în depozit.

## 1. Importuri și foldere de ieșire

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import requests
from requests.exceptions import HTTPError

BASE_URL = "https://raw.githubusercontent.com/hudl/open-data/master/data"
OUTPUT_ROOT = Path("data/raw/statsbomb")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "IAPD-football-data-project/1.0"})


## 2. Descarcă lista competițiilor și sezoanelor

In [ ]:
def get_json(path: str):
    response = session.get(f"{BASE_URL}/{path.lstrip('/')}", timeout=45)
    try:
        response.raise_for_status()
    except HTTPError as exc:
        raise RuntimeError(f"GitHub raw a întors HTTP {response.status_code} pentru {path}") from exc
    return response.json()

competitions = get_json("competitions.json")
competitions_df = pd.json_normalize(competitions, sep=".")
competitions_df.to_csv(OUTPUT_ROOT / "competitions.csv", index=False, encoding="utf-8-sig")
display(competitions_df[[c for c in ["competition_id", "season_id", "competition_name", "season_name", "country_name"] if c in competitions_df.columns]].head(30))


## 3. Alege competiția și sezonul

Valorile de mai jos sunt un exemplu pentru FIFA World Cup 2022. Înlocuiește-le cu ID-urile din tabelul afișat mai sus pentru altă selecție.

In [ ]:
COMPETITION_ID = 43
SEASON_ID = 106

selected = [row for row in competitions if int(row["competition_id"]) == COMPETITION_ID and int(row["season_id"]) == SEASON_ID]
if not selected:
    raise ValueError("Perechea competiție/sezon nu există în competitions.json.")

competition_name = selected[0]["competition_name"]
season_name = selected[0]["season_name"]
print(f"Selecție: {competition_name} — {season_name} ({COMPETITION_ID}:{SEASON_ID})")


## 4. Descarcă meciurile competiției

In [ ]:
matches = get_json(f"matches/{COMPETITION_ID}/{SEASON_ID}.json")

selection_dir = OUTPUT_ROOT / f"competition_{COMPETITION_ID}_season_{SEASON_ID}"
(selection_dir / "events").mkdir(parents=True, exist_ok=True)
(selection_dir / "lineups").mkdir(parents=True, exist_ok=True)

with (selection_dir / "matches_raw.json").open("w", encoding="utf-8") as f:
    json.dump(matches, f, ensure_ascii=False, indent=2)

matches_df = pd.json_normalize(matches, sep=".")
matches_df.to_csv(selection_dir / "matches.csv", index=False, encoding="utf-8-sig")
print(f"Meciuri: {len(matches_df):,}")
display(matches_df.head())


## 5. Descarcă evenimentele și line-up-urile fiecărui meci

Cererea se face secvențial. Fișierele JSON originale sunt păstrate, iar evenimentele și jucătorii din line-up-uri sunt exportați și în CSV.

In [ ]:
all_events = []
all_lineup_players = []
failed_matches = []

for index, match in enumerate(matches, start=1):
    match_id = match["match_id"]
    match_date = match.get("match_date")
    context = {
        "match_id": match_id,
        "match_date": match_date,
        "competition_id": COMPETITION_ID,
        "competition_name": competition_name,
        "season_id": SEASON_ID,
        "season_name": season_name,
    }
    try:
        events = get_json(f"events/{match_id}.json")
        lineups = get_json(f"lineups/{match_id}.json")
        with (selection_dir / "events" / f"{match_id}.json").open("w", encoding="utf-8") as f:
            json.dump(events, f, ensure_ascii=False, indent=2)
        with (selection_dir / "lineups" / f"{match_id}.json").open("w", encoding="utf-8") as f:
            json.dump(lineups, f, ensure_ascii=False, indent=2)
        for event in events:
            all_events.append({**context, **event})
        for team_lineup in lineups:
            team_context = {**context, "team_id": team_lineup.get("team_id"), "team_name": team_lineup.get("team_name")}
            for player in team_lineup.get("lineup", []) or []:
                all_lineup_players.append({**team_context, **player})
        print(f"{index}/{len(matches)} — meci {match_id}: {len(events)} evenimente")
    except Exception as exc:
        failed_matches.append({"match_id": match_id, "error": str(exc)})
        print(f"Avertisment: meciul {match_id} nu a fost descărcat: {exc}")
    time.sleep(0.15)

events_df = pd.json_normalize(all_events, sep=".")
lineups_df = pd.json_normalize(all_lineup_players, sep=".")
events_df.to_csv(selection_dir / "events.csv", index=False, encoding="utf-8-sig")
lineups_df.to_csv(selection_dir / "lineups.csv", index=False, encoding="utf-8-sig")
if failed_matches:
    pd.DataFrame(failed_matches).to_csv(selection_dir / "failed_matches.csv", index=False)

print(f"Rânduri evenimente: {len(events_df):,}; jucători în line-up-uri: {len(lineups_df):,}; meciuri cu erori: {len(failed_matches)}")


## 6. Verifică fișierele și câmpurile

In [ ]:
print("Evenimente:", events_df.shape)
print("Line-up-uri:", lineups_df.shape)
display(events_df.head())
display(lineups_df.head())


### Notă de folosire

Dacă publici sau distribui cercetare bazată pe date, depozitul StatsBomb cere atribuirea sursei StatsBomb și folosirea logo-ului lor. Unele câmpuri, inclusiv minutele individuale jucate, pot necesita derivare din evenimente și line-up-uri și nu sunt furnizate ca tabel statistic agregat în acest depozit. Consultă [README-ul oficial](https://github.com/hudl/open-data) și documentația din depozit înainte de publicare.